# 01 - Probing (Phase 2/3), CPU only  *(optional: reproduces results I already computed)*

**Settings:** Accelerator **None** | Internet **On** | Input: your bundle dataset. No secret needed.
**Runtime:** ~5 min download + ~5 min safety check + ~45-70 min main run.

How it protects you from late failures:
1. A **preflight** (STEP 4) runs the whole pipeline on one layer first (~5 min). If anything is broken you find out then.
2. The main run saves partial results after every group of layers (`results_partial.json`).
3. The last cell **always** packages whatever exists. All files are also in the Output tab under `results/`.

Run with **Save Version -> Save & Run All (Commit)** to run in the background (you can close the tab), or just *Run All*.

In [ ]:
# STEP 1 - locate the uploaded bundle and load the helper module.
import os, sys
from pathlib import Path
_inp = Path(os.environ.get("LF_KAGGLE_INPUT", "/kaggle/input"))
_hits = sorted(_inp.rglob("kaggle_helpers.py"))
assert _hits, ("Bundle not found. In the right panel click 'Add Input' and add the Kaggle Dataset you created "
               "from linguafranca_bundle.zip, then run this cell again.")
sys.path.insert(0, str(_hits[0].parent))
import kaggle_helpers as kh
WORK = kh.setup()

In [ ]:
# STEP 2 - download the Phase-1 hidden states (2.1 GB, public repo, retried automatically).
from lf_common import HF_REPO, HIDDEN_STATES_REVISION
if os.environ.get("LF_TEST_HS"):                       # (local testing only)
    HS = os.environ["LF_TEST_HS"]
else:
    from huggingface_hub import snapshot_download
    kh.download_with_retries(lambda: snapshot_download(
        repo_id=HF_REPO, repo_type="dataset", revision=HIDDEN_STATES_REVISION,
        allow_patterns=["data/hidden_states/*"], local_dir="/kaggle/working/hf", max_workers=8), what="hidden-state download")
    HS = "/kaggle/working/hf/data/hidden_states"
n_files = len([f for f in os.listdir(HS) if f.endswith(".pt")])
print(n_files, "hidden-state files (expected 4078)")
assert n_files >= 4000 or os.environ.get("LF_TEST_HS"), "download incomplete - re-run this cell (it resumes)"

In [ ]:
# STEP 3 - provenance check: rebuild the canonical dataset from the pinned HF revision, compare checksums.
import json, hashlib
rc = kh.run([sys.executable, "scripts/make_canonical_dataset.py", "--out", "/kaggle/working/rebuilt"], "rebuild_dataset")
try:
    bundled = hashlib.sha256(open("data/canonical/augmented_v2.jsonl", "rb").read()).hexdigest()
    rebuilt = json.load(open("/kaggle/working/rebuilt/augmented_v2.stats.json"))["sha256"]
    print("bundled dataset == rebuilt from Hugging Face:", bundled == rebuilt)
except Exception as e:
    print("[info] checksum comparison skipped:", e)

In [ ]:
# STEP 4 - PREFLIGHT (minutes): the full pipeline on ONE layer. Stops here, loudly, if anything is wrong.
PRE = kh.RESULTS / "preflight_probing"
rc = kh.run([sys.executable, "advanced_probing.py", "--data", "data/canonical/augmented_v2.jsonl", "--hs-dir", HS,
             "--out", str(PRE), "--layers", "10"], "preflight_probing")
ok = rc == 0 and (PRE / "results.md").exists() and (PRE / "probes" / "hop1_probe_layer10.joblib").exists()
assert ok, "PREFLIGHT FAILED - see the log above (also in results/logs). Nothing long was started."
print("\nPREFLIGHT PASSED - starting the main run.")

In [ ]:
# STEP 5 - MAIN RUN (all layers by default; set LAYERS = "8,9,10,11,12,13,14" for a ~15 min version).
LAYERS = os.environ.get("LF_TEST_LAYERS", "all")
OUT = kh.RESULTS / "probing"
rc = kh.run_with_retries([sys.executable, "advanced_probing.py", "--data", "data/canonical/augmented_v2.jsonl",
                          "--hs-dir", HS, "--out", str(OUT), "--layers", LAYERS, "--layer-chunk", "7"],
                         "probing", attempts=2)
print("main run exit code:", rc)

In [ ]:
# STEP 6 - sanity checks (each is optional; a failure here does not matter).
kh.run([sys.executable, "evaluate_saved_probe.py", "--probes-dir", str(OUT / "probes"), "--hs-dir", HS,
        "--layers", "8,10,11,13"], "check_new_probes")
kh.run([sys.executable, "evaluate_saved_probe.py", "--legacy", "--hs-dir", HS, "--layers", "0,10,11"], "check_old_probes")
kh.run([sys.executable, "bootstrap_confident_wrong.py", "--hs-dir", HS, "--layers", "8,9,10,11,12,13,14"], "stability")

In [ ]:
# STEP 7 - FINAL (always runs): show + package whatever exists.
kh.show(OUT / "results.md")
kh.package(OUT, "probing_results")
print("\nFiles:", sorted(p.name for p in OUT.glob("*")) if OUT.exists() else "(none)")
print("Download probing_results.zip from the Output tab (or take files from results/probing/).")